
# Giải thích `State` của LIMOncello cho người mới

Notebook này giải thích class `State` trong đoạn code bạn gửi, theo hướng **đọc được code và hiểu công thức phía sau**, chứ không cố chứng minh hình học Lie một cách quá hàn lâm.

Mục tiêu của notebook:

1. Hiểu `state` gồm những gì và vì sao lại dùng manifold.
2. Nối từng hàm `init()`, `predict()`, `f()`, `df_dx()`, `df_dw()`, `update()` với công thức IESKF tương ứng.
3. Chỉ ra những chỗ dễ nhầm nhất cho người mới: `DoFS2`, gravity trên $S^2$, `Adj`, `J_r`, residual point-to-plane, và iterated update.

> **Nguồn chính dùng để viết notebook này**
>
> - đoạn code `State` mà bạn đã gửi
> - paper **LIMOncello** (để đối chiếu Eq. (8), (15), (17), (20))
> - README của repo LIMOncello
> - tài liệu của thư viện `manif`

> **Cách đọc**
>
> Đọc theo thứ tự từ trên xuống. Nếu bạn là người mới hoàn toàn, hãy ưu tiên các phần:
> **Bức tranh lớn → State vector → `predict()` → `update()`**.



## 0. Bức tranh lớn trước khi đọc code

Ta có một hệ LiDAR–IMU. Ý tưởng tổng quát:

- **IMU** chạy rất nhanh, nên nó được dùng để **dự đoán** trạng thái robot liên tục theo thời gian.
- **LiDAR** chạy chậm hơn nhưng cho hình học môi trường, nên nó được dùng để **sửa** sai số drift của IMU bằng các residual point-to-plane.
- Vì trạng thái có **rotation** và **gravity direction**, không thể cộng trừ như vector thường, nên code dùng **manifold**.

Tư duy đúng khi đọc file này là:

- `X` = **nominal state** (trạng thái danh nghĩa) sống trên manifold.
- `P` = **covariance của error-state nhỏ** sống trong tangent space cục bộ.
- `Q` = covariance của process noise IMU.
- `H`, `z`, `R` = Jacobian đo, residual đo và noise đo của LiDAR.

Công thức “xương sống” của file này là:

$$
x = \hat x \oplus \delta x
$$

$$
\hat X_{k+1} = \hat X_k \oplus \big(f(u_k)\,\Delta t\big)
$$

$$
\delta x_{k+1} \approx F_x\,\delta x_k + F_w\,w_k
$$

$$
P_{k+1} = F_x P_k F_x^\top + F_w Q F_w^\top
$$

và khi có LiDAR:

$$
r_i(X) \approx 0
\quad\Rightarrow\quad
\text{lặp nhiều lần để cập nhật } X
$$

Trong notebook này, ký hiệu $\oplus$ / $\ominus$ là cách viết toán học của các hàm như `plus()`, `minus()`, `S2::oplus()`, `S2::ominus()` trong code.



## 1. Phần đầu file: state được định nghĩa như thế nào?

Đây là phần quan trọng nhất của toàn bộ class.

```cpp
struct State {

  using BundleT = manif::Bundle<double,
      manif::SGal3,  // position & rotation & velocity & t
      manif::SE3,    // extrinsics
      manif::R3,     // angular bias
      manif::R3,     // acceleartion bias
      manif::R3      // gravity
  >;

  using Tangent = typename BundleT::Tangent; 

  template<int R = Eigen::Dynamic, int C = R>
  using Mat = Eigen::Matrix<double, R, C>;

  template<int N = Eigen::Dynamic>
  using Vec = Eigen::Matrix<double, N, 1>;


  static constexpr int DoF = BundleT::DoF;  // DoF whole state
  static constexpr int DoFS2 = DoF-1;       // DoF g as S2
  static constexpr int DoFNoise = 4*3;      // b_w, b_a, n_{b_w}, n_{b_a}
  static constexpr int DoFObs = manif::SGal3d::DoF + manif::SE3d::DoF;   // DoF obsevation equation

  BundleT X;
  Mat<DoFS2> P;
  Mat<DoFNoise> Q;

  Vec<3> w; // angular velocity (IMU input)
  Vec<3> a; // linear acceleration (IMU input)

  double stamp;
```


### 1.1 `BundleT` nghĩa là gì?

`BundleT` gom nhiều manifold con lại thành **một state chung**:

$$
X =
\Big(
X_{\mathrm{SGal}(3)},
T_{IL},
b_\omega,
b_a,
g
\Big)
$$

Trong notebook sửa lại ký hiệu extrinsics thành **$T_{IL}$** để khớp với code `lidar2imu`, tức là phép biến đổi **từ LiDAR sang IMU** nếu bạn dùng quy ước quen thuộc “$T_{AB}$ biến tọa độ từ frame $B$ sang frame $A$”.

Trong đó:

- $X_{\mathrm{SGal}(3)}$ chứa **position, velocity, rotation, time**
- $T_{IL}$ là extrinsics LiDAR-to-IMU trên $SE(3)$
- $b_\omega$ là gyro bias
- $b_a$ là accel bias
- $g$ là gravity

Nói đời thường: state này không chỉ chứa pose, mà còn chứa **vận tốc**, **thời gian nội tại**, **ngoại chuẩn LiDAR–IMU**, và **các bias IMU**.

### 1.2 Vì sao `DoF = 25` nhưng `DoFS2 = 24`?

Ý quan trọng nhất là **gravity được lưu như vector 3 chiều**, nhưng **sai số của gravity chỉ được cập nhật theo 2 chiều** vì gravity direction nằm trên mặt cầu $S^2$.

- Trong `X`, gravity được lưu như `manif::R3d(...)` → trông như 3 chiều.
- Nhưng trong bộ lọc, gravity được xử lý như một phần tử trên $S^2$ → error-state của nó chỉ có **2 bậc tự do**.

Vì vậy:

$$
\texttt{DoFS2} = \texttt{DoF} - 1 = 24
$$

Đây là lý do `P` có kích thước `24 \times 24`, không phải `25 \times 25`.

### 1.3 Vai trò của `SGal(3)`

`manif` mô tả `SGal(3)` là Special Galilean group, chứa rotation, translation, velocity và time trong một Lie group thống nhất. Ý tưởng của paper LIMOncello là dùng `SGal(3)` để propagation IMU **gọn hơn và nhất quán hình học hơn** so với biểu diễn tách rời kiểu $SO(3) \times \mathbb R^6$.

Bạn chưa cần thuộc lý thuyết Galilean group. Để đọc được code, chỉ cần nhớ:

- `SGal(3)` chứa **p, v, R, t**
- `plus()` trên `SGal(3)` sẽ cập nhật các thành phần này theo đúng hình học Lie group
- điều đó giúp phần `predict()` ngắn mà vẫn “đúng hình học”

In [1]:

import pandas as pd

state_layout = pd.DataFrame(
    [
        ["SGal3.position", "0..2", 3, "vị trí IMU trong world"],
        ["SGal3.velocity", "3..5", 3, "vận tốc IMU"],
        ["SGal3.rotation", "6..8", 3, "sai số quay địa phương"],
        ["SGal3.time", "9", 1, "biến thời gian nội tại của SGal(3)"],
        ["SE3 extrinsics", "10..15", 6, "LiDAR-to-IMU extrinsics"],
        ["gyro bias", "16..18", 3, "b_w"],
        ["accel bias", "19..21", 3, "b_a"],
        ["gravity (stored)", "22..24", 3, "vector gravity được lưu"],
        ["gravity error-state", "22..23", 2, "gravity direction trên S² trong P"],
    ],
    columns=["Khối state", "Index", "Số chiều", "Ý nghĩa"],
)
state_layout


,Khối state,Index,Số chiều,Ý nghĩa
0,SGal3.position,0..2,3,vị trí IMU trong world
1,SGal3.velocity,3..5,3,vận tốc IMU
2,SGal3.rotation,6..8,3,sai số quay địa phương
3,SGal3.time,9,1,biến thời gian nội tại của SGal(3)
4,SE3 extrinsics,10..15,6,LiDAR-to-IMU extrinsics
5,gyro bias,16..18,3,b_w
6,accel bias,19..21,3,b_a
7,gravity (stored),22..24,3,vector gravity được lưu
8,gravity error-state,22..23,2,gravity direction trên S² trong P



## 2. `init()`: dựng prior ban đầu cho bộ lọc

Trước khi prediction hay update được chạy, bộ lọc cần một trạng thái ban đầu `X`, một covariance ban đầu `P`, và một process-noise covariance `Q`.

Đây là hàm `init()`:


```cpp
void init() {
  
    Config& cfg = Config::getInstance();
    
    // Set initial state
    auto extrinsics = cfg.sensors.extrinsics;
    auto lidar2imu = extrinsics.imu2baselink.inverse() * extrinsics.lidar2baselink;
                                                                //                  Tangent (idx)
    X = BundleT(manif::SGal3d(extrinsics.imu2baselink.translation(),     //                    0
                              Eigen::Quaterniond(extrinsics.imu2baselink.linear()),         // 6
                              {0., 0., 0.},                     // vx, vy, vz                  3
                              0.),                              // delta t                     9
                manif::SE3d(lidar2imu),                         // isometry                   10
                manif::R3d(cfg.sensors.intrinsics.gyro_bias),   // b_w                        16
                manif::R3d(cfg.sensors.intrinsics.accel_bias),  // b_a                        19
                manif::R3d(Vec<3>::UnitZ()                      // g                          22
                           * extrinsics.gravity));

    P.setIdentity();
    P *= 0.01;

    P.diagonal().segment(0, 3).setConstant(cfg.ikfom.covariance.initial_cov.position);
    P.diagonal().segment(6, 3).setConstant(cfg.ikfom.covariance.initial_cov.rotation);
    P.diagonal().segment(3, 3).setConstant(cfg.ikfom.covariance.initial_cov.velocity);
    P.diagonal().segment(16, 3).setConstant(cfg.ikfom.covariance.initial_cov.gyro_bias);
    P.diagonal().segment(19, 3).setConstant(cfg.ikfom.covariance.initial_cov.accel_bias);
    P.diagonal().segment(22, 2).setConstant(cfg.ikfom.covariance.initial_cov.gravity);


    w.setZero();
    a.setZero();

    // Control signal noise covariance (never changes)
    Q.setZero();
 
    Q.block<3, 3>(0, 0) = cfg.ikfom.covariance.gyro       * Eigen::Matrix3d::Identity(); // n_w
    Q.block<3, 3>(3, 3) = cfg.ikfom.covariance.accel      * Eigen::Matrix3d::Identity(); // n_a
    Q.block<3, 3>(6, 6) = cfg.ikfom.covariance.bias_gyro  * Eigen::Matrix3d::Identity(); // n_{b_w}
    Q.block<3, 3>(9, 9) = cfg.ikfom.covariance.bias_accel * Eigen::Matrix3d::Identity(); // n_{b_a}
  }
```

### 2.1 `init()` đang làm gì theo ngôn ngữ toán?

#### a) Khởi tạo state danh nghĩa

Ta có thể viết ý tưởng của phần khởi tạo như sau:

$$
X_0 =
\Big(
X^{(0)}_{\mathrm{SGal}(3)},
T_{IL}^{(0)},
b_{\omega}^{(0)},
b_a^{(0)},
g^{(0)}
\Big)
$$

Trong code:

- `SGal3d(...)` được khởi tạo từ pose IMU ban đầu, vận tốc bằng 0, và `t = 0`
- `SE3d(lidar2imu)` là extrinsics LiDAR-to-IMU
- gyro bias và accel bias lấy từ config
- gravity được đặt theo trục `UnitZ()` nhân với độ lớn gravity trong config

#### b) Khởi tạo covariance ban đầu

Ở đây notebook cũ viết hơi “quá tròn trịa”. Công thức đúng với **chính đoạn code này** là:

1. Trước hết:
   $$
   P_0 \leftarrow 0.01\,I_{24}
   $$

2. Sau đó code **ghi đè** một số block đường chéo:

$$
P_0 =
\operatorname{diag}\!\Big(
\lambda_p I_3,\,
\lambda_v I_3,\,
\lambda_R I_3,\,
0.01,\,
0.01 I_6,\,
\lambda_{b_\omega} I_3,\,
\lambda_{b_a} I_3,\,
\lambda_g I_2
\Big)
$$

nếu ta đọc đúng theo thứ tự tangent của filter:

$$
[\rho,\ \nu,\ \theta,\ s,\ \xi_{IL},\ \delta b_\omega,\ \delta b_a,\ \delta g_{S^2}]
$$

với:

- $\rho \in \mathbb R^3$: position part
- $\nu \in \mathbb R^3$: velocity part
- $\theta \in \mathbb R^3$: rotation part
- $s \in \mathbb R$: time part của `SGal(3)`
- $\xi_{IL} \in \mathbb R^6$: extrinsics `SE3`
- $\delta g_{S^2} \in \mathbb R^2$: gravity error trên $S^2$

Ở đây $\lambda_p,\lambda_v,\lambda_R,\lambda_{b_\omega},\lambda_{b_a},\lambda_g$ chính là **các giá trị đường chéo lấy từ config**.  
Điểm cần nhớ: **time index `9` và extrinsics `10..15` không bị override trong `init()`**, nên chúng giữ giá trị mặc định `0.01` trên đường chéo.

#### c) Khởi tạo process noise

`Q` là covariance của process noise IMU:

$$
Q = \operatorname{diag}
\Big(
q_{n_\omega} I_3,\,
q_{n_a} I_3,\,
q_{n_{b_\omega}} I_3,\,
q_{n_{b_a}} I_3
\Big)
$$

Ứng với:

- `n_w`: gyro measurement noise
- `n_a`: accel measurement noise
- `n_{b_w}`: random walk của gyro bias
- `n_{b_a}`: random walk của accel bias

### 2.2 Điều nên nhớ

`init()` không “tính toán Kalman” gì cả. Nó chỉ trả lời ba câu hỏi:

1. Robot đang bắt đầu ở đâu?
2. Mình tin prior đó đến mức nào?
3. Noise IMU mạnh cỡ nào?


## 3. `f()` và `interpolate_to()`: mô hình động học nominal

`f()` là hàm mô tả **nominal dynamics** – tức state sẽ tiến triển thế nào nếu tin vào IMU hiện tại.

### 3.1 `f()`


```cpp
Tangent f(const Vec<3>& lin_acc, const Vec<3>& ang_vel) {

    Tangent u = Tangent::Zero();
    u.element<0>().coeffs() << 0., 0., 0., 
                               lin_acc - b_a() /* -n_a */ - R().transpose()*g(),
                               ang_vel - b_w() /* -n_w */,
                               1.;
    // u.element<3>().coeffs() = n_{b_w} 
    // u.element<4>().coeffs() = n_{b_a}
    
    return u;
  }
```

### 3.2 Đọc công thức từ `f()`

Phần SGal3 tangent được gán theo đúng thứ tự tangent của `manif` cho `SGal(3)` là $(\rho,\nu,\theta,s)$:

$$
u =
\begin{bmatrix}
0_{3} \\
a_m - b_a - R^\top g \\
\omega_m - b_\omega \\
1
\end{bmatrix}
$$

trong đó:

- $a_m$ là `lin_acc`
- $\omega_m$ là `ang_vel`
- $b_a$ là accel bias
- $b_\omega$ là gyro bias
- $R$ là rotation của robot
- $g$ là gravity trong world, được đưa về body bởi $R^\top g$

### 3.3 Vì sao 3 phần tử đầu bằng 0 mà vị trí vẫn thay đổi?

Đây là chỗ người mới dễ thấy lạ nhất.

Trong notebook cũ mình giải thích đúng ý nhưng còn hơi lỏng. Viết chặt hơn theo `manif` thì với tangent SGal:

$$
\xi = (\rho,\nu,\theta,s)
$$

thư viện `manif` cài đặt phép mũ dưới dạng:

$$
\exp(\xi)=
\Big(
J_l(\theta)\rho + E(\theta)\,s\,\nu,\;
\exp(\theta^\wedge),\;
J_l(\theta)\nu,\;
s
\Big)
$$

Ở đây code đang dùng:

- $\rho = 0$
- $\nu = a_m - b_a - R^\top g$
- $\theta = \omega_m - b_\omega$
- $s = \Delta t$

nên translation của increment vẫn là:

$$
J_l(\theta)\rho + E(\theta)\,s\,\nu
=
E(\theta)\,\Delta t\,\nu
$$

và vì thế **vị trí vẫn thay đổi dù block đầu của tangent bằng 0**.

Nói ngắn gọn: trong `SGal(3)`, **translation increment không chỉ phụ thuộc vào $\rho$ mà còn phụ thuộc vào coupling giữa velocity part và time part**.

### 3.4 Chú ý về dấu và convention

Công thức trong notebook bám theo đúng convention của code:

$$
a_m - b_a - R^\top g
$$

Trong tài liệu IMU, bạn có thể gặp convention hơi khác dấu tùy cách định nghĩa “specific force”. Khi đọc file này, hãy ưu tiên **theo convention của code**, không cố ép nó về một convention khác.


### 3.5 `interpolate_to()`

Hàm này rất ngắn:


```cpp
void interpolate_to(const double& t) {
    double dt = t - this->stamp;
    assert(dt >= 0);

    X = X.plus(f(a, w) * dt);
  }
```


Ý nghĩa:

$$
X(t) \approx X(t_0) \oplus \big(f(a,w)(t-t_0)\big)
$$

Hàm này **chỉ nội suy state**, không cập nhật covariance. Trong pipeline LIO, kiểu hàm này thường được dùng để **deskew point cloud**: mỗi điểm LiDAR được gán với pose gần đúng tại đúng thời điểm nó được bắn ra.



## 4. `df_dx()` và `df_dw()`: Jacobian của động học

Nếu `f()` cho biết state chuyển động ra sao, thì `df_dx()` và `df_dw()` cho biết:

- state hiện tại ảnh hưởng thế nào đến động học
- noise ảnh hưởng thế nào đến động học

### 4.1 `df_dx()`


```cpp
Mat<DoF> df_dx(const Imu& imu) {
    Mat<DoF> out = Mat<DoF>::Zero();

    // velocity 
    out.block<3, 3>(3,  6) = -manif::skew(R().transpose()*g()); // w.r.t R := d(R^-1*g)/dR * d(R^-1)/dR
    out.block<3, 3>(3, 19) = -Mat<3>::Identity(); // w.r.t b_a 
    out.block<3, 3>(3, 22) = -R().transpose(); // w.r.t g
    // rotation
    out.block<3, 3>(6, 16) = -Mat<3>::Identity(); // w.r.t b_w

    return out;
  }
```

### 4.2 Giải thích từng block trong `df_dx()`

Trong code, thành phần quan trọng là:

$$
\dot v = a_m - b_a - R^\top g
$$

Từ đây suy ra các đạo hàm chính.

#### a) Đạo hàm theo rotation

Code:

```cpp
out.block<3, 3>(3,  6) = -manif::skew(R().transpose()*g());
```

Tức là:

$$
\frac{\partial \dot v}{\partial \delta \theta}
=
-\big[ R^\top g \big]_\times
$$

Giải thích trực giác:

- nếu robot quay lệch một chút, vector gravity khi đổi sang body frame cũng đổi
- sự thay đổi đó tạo ra sai số trong gia tốc dùng để cập nhật vận tốc
- ma trận skew xuất hiện vì đạo hàm của rotation perturbation luôn dẫn đến tích có hướng

Ở đây notebook cũ bị **sai dấu** ở bước tuyến tính hóa trung gian. Với **right perturbation** của `manif`:

$$
R' = R\exp(\delta\theta^\wedge)
$$

thì:

$$
(R')^\top g
=
\exp(-\delta\theta^\wedge)R^\top g
\approx
R^\top g + [R^\top g]_\times \delta\theta
$$

vì vậy:

$$
-\,(R')^\top g
\approx
-\,R^\top g - [R^\top g]_\times \delta\theta
$$

nên đúng là:

$$
\frac{\partial \dot v}{\partial \delta\theta}
=
-\,[R^\top g]_\times
$$

khớp với code.

#### b) Đạo hàm theo accel bias

Code:

```cpp
out.block<3, 3>(3, 19) = -I;
```

vì:

$$
\frac{\partial \dot v}{\partial b_a} = -I
$$

Bias accel tăng một chút thì gia tốc thật dùng để propagation giảm tương ứng.

#### c) Đạo hàm theo gravity

Code:

```cpp
out.block<3, 3>(3, 22) = -R().transpose();
```

vì:

$$
\frac{\partial \dot v}{\partial g} = -R^\top
$$

#### d) Đạo hàm của rotation theo gyro bias

Code:

```cpp
out.block<3, 3>(6, 16) = -I;
```

vì:

$$
\dot \theta \sim \omega_m - b_\omega
\quad\Rightarrow\quad
\frac{\partial \dot \theta}{\partial b_\omega} = -I
$$

### 4.3 `df_dw()`

```cpp
Mat<DoF, DoFNoise> df_dw() {
    // w = (n_w, n_a, n_{b_w}, n_{b_a})
    Mat<DoF, DoFNoise> out = Mat<DoF, DoFNoise>::Zero();

    out.block<3, 3>( 6, 0) = -Mat<3>::Identity(); // w.r.t n_w
    out.block<3, 3>( 3, 3) = -Mat<3>::Identity(); // w.r.t n_a
    out.block<3, 3>(16, 6) =  Mat<3>::Identity(); // w.r.t n_{b_w}
    out.block<3, 3>(19, 9) =  Mat<3>::Identity(); // w.r.t n_{b_a}
    
    return out;
  }
```


### 4.4 Giải thích `df_dw()`

Code đang dùng vector noise:

$$
w =
\begin{bmatrix}
n_\omega \\
n_a \\
n_{b_\omega} \\
n_{b_a}
\end{bmatrix}
$$

và inject chúng vào động học như sau:

- noise gyro đi vào phương trình quay
- noise accel đi vào phương trình vận tốc
- random walk đi vào bias gyro
- random walk đi vào bias accel

Đó là lý do ta thấy các block:

$$
\frac{\partial \dot \theta}{\partial n_\omega} = -I,
\qquad
\frac{\partial \dot v}{\partial n_a} = -I,
\qquad
\frac{\partial \dot b_\omega}{\partial n_{b_\omega}} = I,
\qquad
\frac{\partial \dot b_a}{\partial n_{b_a}} = I.
$$

Đây chính là ma trận “noise injection”.



## 5. `predict()`: propagation bằng IMU trên manifold

Đây là nơi diễn ra **prediction step** của IESKF.


```cpp
void predict(const Imu& imu, const double& dt) {
PROFC_NODE("predict")

    Mat<DoF> Adj, Jr; // Adjoint_X(u)^{-1}, J_r(u)  Sola-18, [https://arxiv.org/abs/1812.01537]
    BundleT X_tmp = X.plus(f(imu.lin_accel, imu.ang_vel) * dt, Adj, Jr);
    
    // S2 particular cases. No increment for g
      Mat<3> AdjS2, JrS2;
      S2::boxplus(g(), {0., 0., 0.}, AdjS2, JrS2);

      Adj.template bottomRightCorner<3, 3>() = AdjS2;
      Jr.template bottomRightCorner<3, 3>() = JrS2;

      // Leftmost Jacobian
      Mat<2, 3> Jx;
      S2::ominus(g(), g(), Jx);

      Mat<DoFS2, DoF> left = Mat<DoFS2, DoF>::Identity();
      left.template bottomRightCorner<2, 3>() = Jx;
      
      // Rightmost Jacobian
      Mat<3, 2> Ju;
      S2::oplus(g(), {0., 0.}, {}, Ju);

      Mat<DoF, DoFS2> right = Mat<DoF, DoFS2>::Identity();
      right.template bottomRightCorner<3, 2>() = Ju;

    Mat<DoFS2>           Fx = left * (Adj + Jr * df_dx(imu) * dt) * right; // Pérez-Ruiz-2026 [https://arxiv.org/abs/2512.19567] Eq. (8a)
    Mat<DoFS2, DoFNoise> Fw = left * Jr * df_dw() * dt;                    // Pérez-Ruiz-2026 [https://arxiv.org/abs/2512.19567] Eq. (8b)

    P = Fx * P * Fx.transpose() + Fw * Q * Fw.transpose(); 

    X = X_tmp;

    // Save info
    a = imu.lin_accel;
    w = imu.ang_vel;

    stamp = imu.stamp;
  }
```


### 5.1 Bộ khung toán học của `predict()`

Phần này khớp với logic quen thuộc của error-state filter:

1. Tạo increment từ IMU:
   $$
   u_k = f(a_k, \omega_k)
   $$

2. Đẩy nominal state đi tiếp:
   $$
   X_{k+1}^- = X_k \oplus (u_k \Delta t)
   $$

3. Linearize error-state:
   $$
   \delta x_{k+1} \approx F_x\,\delta x_k + F_w\,w_k
   $$

4. Cập nhật covariance:
   $$
   P_{k+1}^- = F_x P_k F_x^\top + F_w Q F_w^\top
   $$

### 5.2 `Adj` và `Jr` là gì?

Nếu state chỉ là vector Euclid, bạn thường viết gần đúng:

$$
F_x \approx I + A\Delta t
$$

Nhưng ở đây state nằm trên Lie group / manifold, nên khi linearize phải thêm các “phần bù hình học”:

- `Adj`: Adjoint
- `Jr`: right Jacobian

Bạn có thể hiểu nôm na:

- `Adj` giúp **chuyển tangent perturbation giữa các điểm khác nhau trên group**
- `Jr` giúp **linearize một increment hữu hạn** thay vì chỉ một vi phân cực nhỏ

Vì vậy code dùng:

$$
F_x = L\,\big(\mathrm{Adj} + J_r\,\frac{\partial f}{\partial x}\,\Delta t\big)\,R
$$

$$
F_w = L\,J_r\,\frac{\partial f}{\partial w}\,\Delta t
$$

Ở đây mình viết `L` và `R` để chỉ hai ma trận `left` và `right` trong code.

### 5.3 Vì sao có `left` và `right`?

Đây là hệ quả trực tiếp của chuyện gravity nằm trên $S^2$:

- state đầy đủ lưu gravity như 3 chiều
- error-state của gravity chỉ có 2 chiều

Do đó ta cần hai phép chiếu/inject:

- `left`: từ tangent “lưu trữ 25 chiều” sang error-state 24 chiều
- `right`: từ error-state 24 chiều bơm ngược về tangent tương thích với `plus()`

Nói ngắn gọn:

$$
25 \text{ chiều (storage tangent)} \leftrightarrow 24 \text{ chiều (filter tangent)}
$$

### 5.4 Dòng nào ứng với công thức nào?

- `X.plus(f(...) * dt, Adj, Jr)`  
  → propagation nominal state, đồng thời lấy Jacobian hình học của phép update

- `df_dx(imu)`  
  → Jacobian động học theo state

- `df_dw()`  
  → Jacobian động học theo process noise

- `Fx = ...`  
  → ma trận chuyển error-state

- `Fw = ...`  
  → ma trận noise injection rời rạc

- `P = Fx * P * Fx.transpose() + Fw * Q * Fw.transpose();`  
  → propagation covariance

### 5.5 `predict()` lưu gì lại?

Cuối hàm:

```cpp
a = imu.lin_accel;
w = imu.ang_vel;
stamp = imu.stamp;
```

được lưu để về sau có thể `interpolate_to(t)` cho deskewing hoặc đồng bộ thời gian.


In [2]:

import pandas as pd

matrix_shapes = pd.DataFrame(
    [
        ["X", "state danh nghĩa", "(25 stored parameters)"],
        ["P", "covariance error-state", "(24, 24)"],
        ["Q", "process noise covariance", "(12, 12)"],
        ["Fx", "state transition", "(24, 24)"],
        ["Fw", "noise injection", "(24, 12)"],
        ["H", "measurement Jacobian", "(N_valid, 16)"],
        ["z", "measurement residual", "(N_valid, 1)"],
    ],
    columns=["Ký hiệu", "Vai trò", "Kích thước"],
)
matrix_shapes


,Ký hiệu,Vai trò,Kích thước
0,X,state danh nghĩa,(25 stored parameters)
1,P,covariance error-state,"(24, 24)"
2,Q,process noise covariance,"(12, 12)"
3,Fx,state transition,"(24, 24)"
4,Fw,noise injection,"(24, 12)"
5,H,measurement Jacobian,"(N_valid, 16)"
6,z,measurement residual,"(N_valid, 1)"



## 6. `update()`: LiDAR sửa prediction bằng residual point-to-plane

`update()` là phần dài nhất vì nó gồm **hai lớp logic**:

1. Xây mô hình đo `h_model` từ point cloud và map
2. Chạy **iterated** error-state update cho đến hội tụ

### 6.1 Phần xây measurement model `h_model`


```cpp
void update(PointCloudT::Ptr& cloud, charlie::Octree& map) {
PROFC_NODE("update")

    Config& cfg = Config::getInstance();

// OBSERVATION MODEL

    auto h_model = [&](const State& s,
                       Mat<Eigen::Dynamic, DoFObs>& H,
                       Mat<Eigen::Dynamic, 1>&      z) {

      int N = cloud->size();

      std::vector<bool> chosen(N, false);
      Planes planes(N);

      std::vector<int> indices(N);
      std::iota(indices.begin(), indices.end(), 0);
      
      std::for_each(
        std::execution::par_unseq,
        indices.begin(),
        indices.end(),
        [&](int i) {
          PointT pt = cloud->points[i];
          Vec<3> p = pt.getVector3fMap().cast<double>();
          Vec<3> g = s.isometry() * s.L2I_isometry() * p; // global coords 

          std::vector<pcl::PointXYZ> neighbors;
          std::vector<float> pointSearchSqDis;
          map.knn(pcl::PointXYZ(g(0), g(1), g(2)),
                  cfg.ikfom.plane.points,
                  neighbors,
                  pointSearchSqDis);
          
          if (neighbors.size() < cfg.ikfom.plane.points 
              or pointSearchSqDis.back() > cfg.ikfom.plane.max_sqrt_dist)
                return;
          
          Eigen::Vector4d p_abcd = Eigen::Vector4d::Zero();
          if (not estimate_plane(p_abcd, neighbors, cfg.ikfom.plane.plane_threshold))
            return;


          chosen[i] = true;
          planes[i] = Plane(p, p_abcd);
        }
      ); // end for_each

      Planes valid_planes;

      for (int i = 0; i < N; i++) {
        if (chosen[i])
          valid_planes.push_back(planes[i]);        
      }

      H = Mat<>::Zero(valid_planes.size(), DoFObs);
      z = Mat<>::Zero(valid_planes.size(), 1);

      indices.resize(valid_planes.size());
      std::iota(indices.begin(), indices.end(), 0);

      // For each plane, calculate its derivative and distance
      std::for_each(
        std::execution::par_unseq,
        indices.begin(),
        indices.end(),
        [&](int i) {
          Plane m = valid_planes[i];

          // Differentiate w.r.t. SGal3
          Mat<3, manif::SGal3d::DoF> J_s;
          Vec<3> g = s.X.element<0>().act(s.L2I_isometry() * m.p, J_s);

          H.block<1, manif::SGal3d::DoF>(i, 0) << m.n.head(3).transpose() * J_s;

          // Differentiate w.r.t. SE3
          if (cfg.ikfom.estimate_extrinsics) {
            Eigen::Matrix<double, 3, manif::SE3d::DoF> J_e;
            manif::SE3d(isometry() * L2I_isometry()).act(m.p, J_e);
            
            H.block<1, manif::SE3d::DoF>(i, manif::SGal3d::DoF) << m.n.head(3).transpose() * J_e;
          }

          z(i) = -dist2plane(m.n, g);
        }
      );

    }; // end h_model
```

### 6.2 `h_model` đang làm gì?

Với mỗi điểm LiDAR $p_i^L$ trong cloud:

1. Đổi sang IMU frame:
   $$
   p_i^I = T_{IL}\,p_i^L
   $$

2. Đổi sang world:
   $$
   p_i^W = X_{\mathrm{SGal}(3)} \cdot p_i^I
   $$

   hay viết gộp:
   $$
   p_i^W = T_{WI}\,T_{IL}\,p_i^L
   $$

3. Tìm `k` láng giềng gần nhất trong map quanh $p_i^W$

4. Fit một mặt phẳng cục bộ từ các láng giềng đó

5. Tạo residual point-to-plane và Jacobian của residual theo state

### 6.3 Residual point-to-plane được hiểu thế nào?

Nếu mặt phẳng cục bộ có dạng:

$$
\pi_i = (n_i, d_i),
\qquad
n_i^\top x + d_i = 0
$$

thì residual về bản chất là **signed distance** từ điểm world $p_i^W$ đến mặt phẳng:

$$
r_i(X) \approx n_i^\top p_i^W + d_i
$$

Trong code:

```cpp
z(i) = -dist2plane(m.n, g);
```

nghĩa là `z` được lấy bằng **âm của residual** theo convention nội bộ. Dấu âm này không làm đổi bản chất bài toán; nó chỉ đổi hướng update.

> Lưu ý: vì ta không thấy định nghĩa `dist2plane()` trong đoạn code này, công thức trên nên hiểu là “bản chất của residual”, có thể có chuẩn hóa theo norm của normal tùy implementation.

### 6.4 Jacobian `H` được dựng thế nào?

Code:

```cpp
Vec<3> g = s.X.element<0>().act(s.L2I_isometry() * m.p, J_s);
H.block<1, manif::SGal3d::DoF>(i, 0) << m.n.head(3).transpose() * J_s;
```

đây là chain rule:

$$
\frac{\partial r_i}{\partial \delta x_{\mathrm{SGal3}}}
=
n_i^\top
\frac{\partial p_i^W}{\partial \delta x_{\mathrm{SGal3}}}
=
n_i^\top J_s
$$

Viết chặt hơn theo đúng `manif::SGal3::act`, với thứ tự tangent $(\rho,\nu,\theta,s)$:

$$
J_s
=
\begin{bmatrix}
R_{WI} & 0_{3\times 3} & -R_{WI}[p_i^I]_\times & v_{WI}
\end{bmatrix}
$$

trong đó:

- $p_i^I = T_{IL}p_i^L$
- $R_{WI}$ là rotation của IMU trong world
- $v_{WI}$ là velocity hiện tại của IMU trong world

Nghĩa là:

- 3 cột **position** tác động trực tiếp
- 3 cột **velocity** của `SGal3` ở Jacobian `act()` bằng **0**
- 3 cột **rotation** tác động qua $-[p_i^I]_\times$
- cột **time** tác động qua vector vận tốc hiện tại $v_{WI}$

Vì vậy notebook cũ viết “LiDAR residual phụ thuộc trực tiếp vào pose / velocity / time trong `SGal3`” là **quá lỏng**. Viết chính xác hơn là:

- residual phụ thuộc trực tiếp vào **position**, **rotation**, và **time perturbation** của `SGal3`
- còn 3 cột **velocity perturbation** trong Jacobian `J_s` bằng 0
- velocity vẫn có thể ảnh hưởng gián tiếp qua propagation và qua cột thời gian

Tương tự với extrinsics:

$$
\frac{\partial r_i}{\partial \delta x_{SE3}}
=
n_i^\top J_e
$$

Code chỉ dựng `H` cho phần `SGal3` và `SE3`, nên:

$$
\texttt{DoFObs} = \texttt{SGal3::DoF} + \texttt{SE3::DoF} = 10 + 6 = 16
$$

Các bias IMU và gravity **không được đo trực tiếp** trong `H`; chúng được sửa **gián tiếp thông qua covariance và tương quan chéo**.

### 6.5 Một chỗ cực dễ nhầm

Trong lambda này, biến cục bộ tên `g` là:

```cpp
Vec<3> g = s.isometry() * s.L2I_isometry() * p; // global coords
```

Tức là **global point**, không phải gravity.  
Cùng tên `g`, nhưng ngữ nghĩa khác hoàn toàn với getter `g()` của state.


### 6.6 Phần iterated update


```cpp
// IESEKF UPDATE

    BundleT    X_predicted = X;
    Mat<DoFS2> P_predicted = P;

    Mat<Eigen::Dynamic, DoFObs> H;
    Mat<Eigen::Dynamic, 1>      z;
    Mat<DoFS2> KH;

    double R = cfg.ikfom.lidar_noise;

    Vec<3> g_pred = X_predicted.element<4>().coeffs();

    int i(0);

    do {
      h_model(*this, H, z); // Update H,z and set K to zeros

      // project P to homemorphic space
        Mat<DoF> J_;
        Vec<DoFS2> dx = X.minus(X_predicted, J_).coeffs().head(DoFS2);
        dx.tail(2) = S2::ominus(g(), g_pred);

        // d/db ((g oplus b) ominus g_pred) | b = 0
        Mat<DoFS2> J_inv = J_.topLeftCorner(DoFS2, DoFS2).inverse();
        P = J_inv * P * J_inv.transpose(); // !! projection

      // Build K from blocks (numerical stability)
        Mat<DoFObs> HTH = H.transpose() * H / R;
        
        Mat<DoFS2>  P_inv = P.inverse();
        P_inv.template topLeftCorner<DoFObs, DoFObs>() += HTH;
        P_inv = P_inv.inverse();

        Vec<DoFS2> Kz = P_inv.template topLeftCorner<DoFS2, DoFObs>() * H.transpose() * z / R;

        KH.setZero();
        KH.template topLeftCorner<DoFS2, DoFObs>() = P_inv.template topLeftCorner<DoFS2, DoFObs>() * HTH;

      dx = Kz + (KH - Mat<DoFS2>::Identity()) * J_inv * dx; 
      
      // Update manif Bundle, left g unmodified
      Tangent tau = Tangent::Zero();
      tau.coeffs().head(DoF-3) = dx.head(DoF-3);

      // Update
      X = X.plus(tau);
      g(S2::oplus(g(), dx.tail(2)));

      if ((dx.array().abs() <= cfg.ikfom.tolerance).all())
        break;

    } while (i++ < cfg.ikfom.max_iters);

    P = (Mat<DoFS2>::Identity() - KH) * P;
    X = X;
```

### 6.7 Ý tưởng toán học của iterated update

Đây là chỗ làm file này trở thành **Iterated** Error-State Kalman Filter, chứ không phải EKF một phát rồi thôi.

Ta bắt đầu từ prior được dự đoán:

$$
X^- ,\; P^-
$$

Sau đó lặp nhiều lần. Ở vòng lặp thứ $\ell$:

1. Re-linearize residual tại state hiện tại:
   $$
   z^{(\ell)},\; H^{(\ell)}
   $$

2. Tính local error giữa state hiện tại và state predicted:
   $$
   \delta x^{(\ell)} = X^{(\ell)} \ominus X^-
   $$
   riêng gravity thì dùng:
   $$
   \delta g^{(\ell)} = g^{(\ell)} \ominus_{S^2} g^-
   $$

3. Chiếu covariance về tangent space đang xét:
   $$
   \widetilde P^{(\ell)} \leftarrow J^{-1} P^{(\ell)} (J^{-1})^\top
   $$

4. Giải một bước MAP / information-form cục bộ.

Ở đây notebook cũ viết:
$$
(P^{-1}+H^\top R^{-1}H)^{-1}
$$
như thể $H$ và $P$ có cùng kích thước, nhưng **đó là sai về kích thước đối với chính code này**:

- `H` có kích thước $N \times 16$
- `P` có kích thước $24 \times 24$

Cách viết đúng là dùng ma trận đo **zero-padded**:

$$
\bar H^{(\ell)} =
\begin{bmatrix}
H^{(\ell)} & 0_{N\times 8}
\end{bmatrix}
\in \mathbb R^{N\times 24}
$$

khi đó:

$$
P_\star^{(\ell)} =
\Big(
(\widetilde P^{(\ell)})^{-1}
+
(\bar H^{(\ell)})^\top R^{-1}\bar H^{(\ell)}
\Big)^{-1}
$$

và các lượng trong code tương ứng với:

$$
Kz = P_\star^{(\ell)}(\bar H^{(\ell)})^\top R^{-1} z^{(\ell)}
$$

$$
KH = P_\star^{(\ell)}(\bar H^{(\ell)})^\top R^{-1}\bar H^{(\ell)}
$$

rồi increment được tính bởi:

$$
dx =
Kz
+
\big(KH - I\big)J^{-1}\delta x^{(\ell)}
$$

5. Cập nhật state:
   $$
   X \leftarrow X \oplus \tau
   $$
   với gravity cập nhật riêng:
   $$
   g \leftarrow g \oplus_{S^2} dx_g
   $$

6. Dừng nếu:
   $$
   |dx| \le \text{tolerance}
   $$

### 6.8 Vì sao code update gravity riêng?

Trong code:

```cpp
Tangent tau = Tangent::Zero();
tau.coeffs().head(DoF-3) = dx.head(DoF-3);

X = X.plus(tau);
g(S2::oplus(g(), dx.tail(2)));
```

nghĩa là:

- mọi thành phần khác được update bằng `Bundle::plus()`
- gravity **không** được cộng thẳng bằng vector 3D
- gravity dùng **2 tham số perturbation trên $S^2$**

Đây chính là hiện thực của ý tưởng:

> **gravity direction có chuẩn cố định, nên chỉ nên sửa hướng của nó trên mặt cầu, không nên update tự do trong $\mathbb R^3$.**

### 6.9 Tại sao công thức Kalman trong code trông khác sách nhập môn?

Trong sách EKF, bạn thường thấy:

$$
K = PH^\top (HPH^\top + R)^{-1}
$$

Còn ở đây code làm việc với:

```cpp
P_inv = P.inverse();
P_inv.topLeftCorner<DoFObs, DoFObs>() += HTH;
P_inv = P_inv.inverse();
```

Cách viết này gần với **information form / MAP step**, và thường ổn định số hơn khi số residual lớn.  
Về bản chất, nó vẫn đang giải cùng một bài toán Gaussian tuyến tính cục bộ, chỉ là code thêm $H^\top R^{-1}H$ vào **khối được đo trực tiếp** của ma trận thông tin.

### 6.10 Covariance cuối cùng

Cuối vòng lặp:

$$
P \leftarrow (I - KH)P
$$

Đây là covariance sau update cục bộ theo linearization cuối cùng.


## 7. Bảng nối nhanh: dòng code ↔ ý nghĩa toán học

| Dòng code | Ý nghĩa |
|---|---|
| `X.plus(f(...) * dt)` | propagation nominal state trên manifold |
| `X.minus(X_predicted, J_)` | local error giữa state hiện tại và predicted |
| `S2::ominus(g(), g_pred)` | local error 2D của gravity trên $S^2$ |
| `df_dx()` | Jacobian của động học theo state |
| `df_dw()` | Jacobian của động học theo process noise |
| `Fx = left * (...) * right` | state transition sau khi chiếu/inject gravity giữa 25D và 24D |
| `P = Fx P Fx^T + Fw Q Fw^T` | propagation covariance |
| `map.knn(...)` | tìm láng giềng gần nhất trong map để fit plane |
| `estimate_plane(...)` | fit mặt phẳng cục bộ |
| `z(i) = -dist2plane(...)` | residual point-to-plane |
| `H = ... n^T J_s ... n^T J_e` | Jacobian residual theo SGal3 và extrinsics |
| `dx = Kz + (KH-I)J_inv dx` | iterated correction step |
| `g(S2::oplus(...))` | update gravity đúng trên manifold $S^2$ |




## 8. Những chỗ người mới dễ “lú”

### 8.1 `P` không phải covariance của toàn bộ `X` theo kiểu vector thường

`X` là state **trên manifold**, còn `P` là covariance của **error-state nhỏ** trong tangent space. Đây là tinh thần lõi của error-state Kalman filter.

### 8.2 `DoFObs` không phải số residual

`DoFObs = 16` là **số chiều state được đo trực tiếp** bởi LiDAR model.  
Số residual thật sự là `valid_planes.size()`.

### 8.3 `R` trong `update()` không phải rotation matrix

```cpp
double R = cfg.ikfom.lidar_noise;
```

Ở đây `R` là **measurement noise scalar**.

### 8.4 `df_dx(const Imu& imu)` không dùng biến `imu`

Điều này hơi lạ với người mới nhưng không sai. Ở implementation hiện tại, Jacobian động học theo state không cần trực tiếp dùng giá trị `imu` trong hàm này; interface được giữ vậy cho đồng nhất.

### 8.5 `P_predicted` hiện tại không được dùng tiếp

Trong đoạn code này:

```cpp
Mat<DoFS2> P_predicted = P;
```

biến đó không được dùng về sau. Đây có thể là phần còn lại sau refactor.

### 8.6 `X = X;` ở cuối hàm là no-op

Dòng này không làm gì cả:

```cpp
X = X;
```

Nó không sai logic, chỉ là thừa.


## 9. 5 công thức quan trọng nhất cần nhớ sau khi đọc xong

Nếu bạn chỉ muốn giữ lại “xương sống” của file này, hãy nhớ 5 công thức sau.

### (1) True state = nominal state + error-state trên manifold
$$
x = \hat x \oplus \delta x
$$

### (2) Nominal propagation bằng IMU
$$
\hat X_{k+1}^- = \hat X_k \oplus \big(f(u_k)\Delta t\big)
$$

### (3) Error-state propagation
$$
\delta x_{k+1} \approx F_x\,\delta x_k + F_w\,w_k
$$

### (4) Covariance propagation
$$
P_{k+1}^- = F_x P_k F_x^\top + F_w Q F_w^\top
$$

### (5) Iterated point-to-plane update
$$
r_i(X) \approx n_i^\top p_i^W + d_i
$$

và trong từng vòng lặp:

$$
P_\star =
\Big(
\widetilde P^{-1} + \bar H^\top R^{-1}\bar H
\Big)^{-1}
$$

$$
dx = P_\star \bar H^\top R^{-1} z + (KH-I)J^{-1}\delta x
$$

$$
X \leftarrow X \oplus \tau,
\qquad
g \leftarrow g \oplus_{S^2} dx_g
$$

Điểm cần nhớ ở đây là:

- `H` thật trong code chỉ có **16 cột**
- để viết công thức gọn trong không gian lỗi 24 chiều, ta phải hiểu nó như ma trận **$\bar H$ đã được zero-pad**
- gravity không đi vào `Bundle::plus()` như các thành phần còn lại, mà được cập nhật riêng trên $S^2$

Đó chính là “bộ xương” của toàn bộ class `State`.


## 10. Phụ lục: toàn bộ đoạn code đang được giải thích

<details>
<summary>Bấm để mở toàn bộ code</summary>

```cpp
#pragma once

#include <execution>
#include <numeric>
#include <algorithm>
#include <boost/circular_buffer.hpp>

#include <Eigen/Dense>
#include <Eigen/Geometry>

#include "Core/Imu.hpp"
#include "Core/Plane.hpp"
#include "Core/Octree.hpp"
#include <Core/S2.hpp>

#include "Utils/Config.hpp"
#include "Utils/PCL.hpp"

#include <manif/manif.h>
#include <manif/SGal3.h>
#include <manif/SE3.h>
#include <manif/Bundle.h>
#include <manif/Rn.h>


struct State {

  using BundleT = manif::Bundle<double,
      manif::SGal3,  // position & rotation & velocity & t
      manif::SE3,    // extrinsics
      manif::R3,     // angular bias
      manif::R3,     // acceleartion bias
      manif::R3      // gravity
  >;

  using Tangent = typename BundleT::Tangent; 

  template<int R = Eigen::Dynamic, int C = R>
  using Mat = Eigen::Matrix<double, R, C>;

  template<int N = Eigen::Dynamic>
  using Vec = Eigen::Matrix<double, N, 1>;


  static constexpr int DoF = BundleT::DoF;  // DoF whole state
  static constexpr int DoFS2 = DoF-1;       // DoF g as S2
  static constexpr int DoFNoise = 4*3;      // b_w, b_a, n_{b_w}, n_{b_a}
  static constexpr int DoFObs = manif::SGal3d::DoF + manif::SE3d::DoF;   // DoF obsevation equation

  BundleT X;
  Mat<DoFS2> P;
  Mat<DoFNoise> Q;

  Vec<3> w; // angular velocity (IMU input)
  Vec<3> a; // linear acceleration (IMU input)

  double stamp;

  State() : stamp(-1.0) {};

  void init() {

    Config& cfg = Config::getInstance();

    // Set initial state
    auto extrinsics = cfg.sensors.extrinsics;
    auto lidar2imu = extrinsics.imu2baselink.inverse() * extrinsics.lidar2baselink;
                                                                //                  Tangent (idx)
    X = BundleT(manif::SGal3d(extrinsics.imu2baselink.translation(),     //                    0
                              Eigen::Quaterniond(extrinsics.imu2baselink.linear()),         // 6
                              {0., 0., 0.},                     // vx, vy, vz                  3
                              0.),                              // delta t                     9
                manif::SE3d(lidar2imu),                         // isometry                   10
                manif::R3d(cfg.sensors.intrinsics.gyro_bias),   // b_w                        16
                manif::R3d(cfg.sensors.intrinsics.accel_bias),  // b_a                        19
                manif::R3d(Vec<3>::UnitZ()                      // g                          22
                           * extrinsics.gravity));

    P.setIdentity();
    P *= 0.01;

    P.diagonal().segment(0, 3).setConstant(cfg.ikfom.covariance.initial_cov.position);
    P.diagonal().segment(6, 3).setConstant(cfg.ikfom.covariance.initial_cov.rotation);
    P.diagonal().segment(3, 3).setConstant(cfg.ikfom.covariance.initial_cov.velocity);
    P.diagonal().segment(16, 3).setConstant(cfg.ikfom.covariance.initial_cov.gyro_bias);
    P.diagonal().segment(19, 3).setConstant(cfg.ikfom.covariance.initial_cov.accel_bias);
    P.diagonal().segment(22, 2).setConstant(cfg.ikfom.covariance.initial_cov.gravity);


    w.setZero();
    a.setZero();

    // Control signal noise covariance (never changes)
    Q.setZero();

    Q.block<3, 3>(0, 0) = cfg.ikfom.covariance.gyro       * Eigen::Matrix3d::Identity(); // n_w
    Q.block<3, 3>(3, 3) = cfg.ikfom.covariance.accel      * Eigen::Matrix3d::Identity(); // n_a
    Q.block<3, 3>(6, 6) = cfg.ikfom.covariance.bias_gyro  * Eigen::Matrix3d::Identity(); // n_{b_w}
    Q.block<3, 3>(9, 9) = cfg.ikfom.covariance.bias_accel * Eigen::Matrix3d::Identity(); // n_{b_a}
  } 

  void predict(const Imu& imu, const double& dt) {
PROFC_NODE("predict")

    Mat<DoF> Adj, Jr; // Adjoint_X(u)^{-1}, J_r(u)  Sola-18, [https://arxiv.org/abs/1812.01537]
    BundleT X_tmp = X.plus(f(imu.lin_accel, imu.ang_vel) * dt, Adj, Jr);

    // S2 particular cases. No increment for g
      Mat<3> AdjS2, JrS2;
      S2::boxplus(g(), {0., 0., 0.}, AdjS2, JrS2);

      Adj.template bottomRightCorner<3, 3>() = AdjS2;
      Jr.template bottomRightCorner<3, 3>() = JrS2;

      // Leftmost Jacobian
      Mat<2, 3> Jx;
      S2::ominus(g(), g(), Jx);

      Mat<DoFS2, DoF> left = Mat<DoFS2, DoF>::Identity();
      left.template bottomRightCorner<2, 3>() = Jx;

      // Rightmost Jacobian
      Mat<3, 2> Ju;
      S2::oplus(g(), {0., 0.}, {}, Ju);

      Mat<DoF, DoFS2> right = Mat<DoF, DoFS2>::Identity();
      right.template bottomRightCorner<3, 2>() = Ju;

    Mat<DoFS2>           Fx = left * (Adj + Jr * df_dx(imu) * dt) * right; // Pérez-Ruiz-2026 [https://arxiv.org/abs/2512.19567] Eq. (8a)
    Mat<DoFS2, DoFNoise> Fw = left * Jr * df_dw() * dt;                    // Pérez-Ruiz-2026 [https://arxiv.org/abs/2512.19567] Eq. (8b)

    P = Fx * P * Fx.transpose() + Fw * Q * Fw.transpose(); 

    X = X_tmp;

    // Save info
    a = imu.lin_accel;
    w = imu.ang_vel;

    stamp = imu.stamp;
  }


  void interpolate_to(const double& t) {
    double dt = t - this->stamp;
    assert(dt >= 0);

    X = X.plus(f(a, w) * dt);
  }


  Tangent f(const Vec<3>& lin_acc, const Vec<3>& ang_vel) {

    Tangent u = Tangent::Zero();
    u.element<0>().coeffs() << 0., 0., 0., 
                               lin_acc - b_a() /* -n_a */ - R().transpose()*g(),
                               ang_vel - b_w() /* -n_w */,
                               1.;
    // u.element<3>().coeffs() = n_{b_w} 
    // u.element<4>().coeffs() = n_{b_a}

    return u;
  }

  Mat<DoF> df_dx(const Imu& imu) {
    Mat<DoF> out = Mat<DoF>::Zero();

    // velocity 
    out.block<3, 3>(3,  6) = -manif::skew(R().transpose()*g()); // w.r.t R := d(R^-1*g)/dR * d(R^-1)/dR
    out.block<3, 3>(3, 19) = -Mat<3>::Identity(); // w.r.t b_a 
    out.block<3, 3>(3, 22) = -R().transpose(); // w.r.t g
    // rotation
    out.block<3, 3>(6, 16) = -Mat<3>::Identity(); // w.r.t b_w

    return out;
  }

  Mat<DoF, DoFNoise> df_dw() {
    // w = (n_w, n_a, n_{b_w}, n_{b_a})
    Mat<DoF, DoFNoise> out = Mat<DoF, DoFNoise>::Zero();

    out.block<3, 3>( 6, 0) = -Mat<3>::Identity(); // w.r.t n_w
    out.block<3, 3>( 3, 3) = -Mat<3>::Identity(); // w.r.t n_a
    out.block<3, 3>(16, 6) =  Mat<3>::Identity(); // w.r.t n_{b_w}
    out.block<3, 3>(19, 9) =  Mat<3>::Identity(); // w.r.t n_{b_a}

    return out;
  }

  void update(PointCloudT::Ptr& cloud, charlie::Octree& map) {
PROFC_NODE("update")

    Config& cfg = Config::getInstance();

// OBSERVATION MODEL

    auto h_model = [&](const State& s,
                       Mat<Eigen::Dynamic, DoFObs>& H,
                       Mat<Eigen::Dynamic, 1>&      z) {

      int N = cloud->size();

      std::vector<bool> chosen(N, false);
      Planes planes(N);

      std::vector<int> indices(N);
      std::iota(indices.begin(), indices.end(), 0);

      std::for_each(
        std::execution::par_unseq,
        indices.begin(),
        indices.end(),
        [&](int i) {
          PointT pt = cloud->points[i];
          Vec<3> p = pt.getVector3fMap().cast<double>();
          Vec<3> g = s.isometry() * s.L2I_isometry() * p; // global coords 

          std::vector<pcl::PointXYZ> neighbors;
          std::vector<float> pointSearchSqDis;
          map.knn(pcl::PointXYZ(g(0), g(1), g(2)),
                  cfg.ikfom.plane.points,
                  neighbors,
                  pointSearchSqDis);

          if (neighbors.size() < cfg.ikfom.plane.points 
              or pointSearchSqDis.back() > cfg.ikfom.plane.max_sqrt_dist)
                return;

          Eigen::Vector4d p_abcd = Eigen::Vector4d::Zero();
          if (not estimate_plane(p_abcd, neighbors, cfg.ikfom.plane.plane_threshold))
            return;


          chosen[i] = true;
          planes[i] = Plane(p, p_abcd);
        }
      ); // end for_each

      Planes valid_planes;

      for (int i = 0; i < N; i++) {
        if (chosen[i])
          valid_planes.push_back(planes[i]);        
      }

      H = Mat<>::Zero(valid_planes.size(), DoFObs);
      z = Mat<>::Zero(valid_planes.size(), 1);

      indices.resize(valid_planes.size());
      std::iota(indices.begin(), indices.end(), 0);

      // For each plane, calculate its derivative and distance
      std::for_each(
        std::execution::par_unseq,
        indices.begin(),
        indices.end(),
        [&](int i) {
          Plane m = valid_planes[i];

          // Differentiate w.r.t. SGal3
          Mat<3, manif::SGal3d::DoF> J_s;
          Vec<3> g = s.X.element<0>().act(s.L2I_isometry() * m.p, J_s);

          H.block<1, manif::SGal3d::DoF>(i, 0) << m.n.head(3).transpose() * J_s;

          // Differentiate w.r.t. SE3
          if (cfg.ikfom.estimate_extrinsics) {
            Eigen::Matrix<double, 3, manif::SE3d::DoF> J_e;
            manif::SE3d(isometry() * L2I_isometry()).act(m.p, J_e);

            H.block<1, manif::SE3d::DoF>(i, manif::SGal3d::DoF) << m.n.head(3).transpose() * J_e;
          }

          z(i) = -dist2plane(m.n, g);
        }
      );

    }; // end h_model

// IESEKF UPDATE

    BundleT    X_predicted = X;
    Mat<DoFS2> P_predicted = P;

    Mat<Eigen::Dynamic, DoFObs> H;
    Mat<Eigen::Dynamic, 1>      z;
    Mat<DoFS2> KH;

    double R = cfg.ikfom.lidar_noise;

    Vec<3> g_pred = X_predicted.element<4>().coeffs();

    int i(0);

    do {
      h_model(*this, H, z); // Update H,z and set K to zeros

      // project P to homemorphic space
        Mat<DoF> J_;
        Vec<DoFS2> dx = X.minus(X_predicted, J_).coeffs().head(DoFS2);
        dx.tail(2) = S2::ominus(g(), g_pred);

        // d/db ((g oplus b) ominus g_pred) | b = 0
        Mat<DoFS2> J_inv = J_.topLeftCorner(DoFS2, DoFS2).inverse();
        P = J_inv * P * J_inv.transpose(); // !! projection

      // Build K from blocks (numerical stability)
        Mat<DoFObs> HTH = H.transpose() * H / R;

        Mat<DoFS2>  P_inv = P.inverse();
        P_inv.template topLeftCorner<DoFObs, DoFObs>() += HTH;
        P_inv = P_inv.inverse();

        Vec<DoFS2> Kz = P_inv.template topLeftCorner<DoFS2, DoFObs>() * H.transpose() * z / R;

        KH.setZero();
        KH.template topLeftCorner<DoFS2, DoFObs>() = P_inv.template topLeftCorner<DoFS2, DoFObs>() * HTH;

      dx = Kz + (KH - Mat<DoFS2>::Identity()) * J_inv * dx; 

      // Update manif Bundle, left g unmodified
      Tangent tau = Tangent::Zero();
      tau.coeffs().head(DoF-3) = dx.head(DoF-3);

      // Update
      X = X.plus(tau);
      g(S2::oplus(g(), dx.tail(2)));

      if ((dx.array().abs() <= cfg.ikfom.tolerance).all())
        break;

    } while (i++ < cfg.ikfom.max_iters);

    P = (Mat<DoFS2>::Identity() - KH) * P;
    X = X;
  }


// Getters
  inline Vec<3>                p() const { return X.element<0>().translation();             }
  inline Mat<3>                R() const { return X.element<0>().quat().toRotationMatrix(); }
  inline Eigen::Quaterniond quat() const { return X.element<0>().quat();                    }
  inline Vec<3>                v() const { return X.element<0>().linearVelocity();          }
  inline double                t() const { return X.element<0>().t();                       }
  inline Vec<3>              b_w() const { return X.element<2>().coeffs();                  }
  inline Vec<3>              b_a() const { return X.element<3>().coeffs();                  }
  inline Vec<3>                g() const { return X.element<4>().coeffs();                  }

  inline Eigen::Isometry3d isometry() const {
    Eigen::Isometry3d T;
    T.linear() = R();
    T.translation() = p();
    return T;
  }

  inline Eigen::Isometry3d L2I_isometry() const {
    return X.element<1>().isometry();
  }

// Setters
  void quat(const Eigen::Quaterniond& in) { X.element<0>() = manif::SGal3d(p(), in, v(), t()); } 
  void b_w (const Vec<3>& in)             { X.element<2>() = manif::R3d(in);                   }
  void b_a (const Vec<3>& in)             { X.element<3>() = manif::R3d(in);                   }
  void g   (const Vec<3>& in)             { X.element<4>() = manif::R3d(in);                   }

};

typedef boost::circular_buffer<State> States;
```

</details>



## 11. Tài liệu tham khảo ngắn

1. **Đoạn code bạn gửi** – class `State`.
2. **Paper LIMOncello** – để đối chiếu các khối công thức:
   - linearization của error-state
   - prediction trên manifold
   - measurement function point-to-plane
   - iterated update / MAP step
3. **README của repo LIMOncello** – giải thích vai trò của `State.hpp` trong toàn bộ dự án.
4. **Tài liệu `manif`** – để hiểu `Bundle<>`, `SGal(3)`, `SE(3)`, `R^3`, và các phép `plus/minus`.

> Gợi ý học tiếp:
>
> - Khi đã quen notebook này, hãy đọc lại riêng hàm `update()` thêm một lần nữa.
> - Chỗ khó nhất thường là: `S2`, `Adj/Jr`, và lý do LiDAR không đo trực tiếp bias nhưng bias vẫn được sửa gián tiếp.



> Bản notebook đã được sửa lại ở các chỗ: ký hiệu extrinsics, covariance khởi tạo, dấu trong tuyến tính hóa $R^\top g$, Jacobian `SGal3::act`, và công thức information-form của `update()`.